Extraindo PDF

In [84]:
!pip install transformers datasets pdfplumber accelerate sentencepiece PyMuPDF torch json re

ERROR: Could not find a version that satisfies the requirement json (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for json


In [85]:
import json
import re
# import pdfplumber
import fitz  # PyMuPDF
import re
import torch
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

In [ ]:

def extract_text_from_pdf_pymupdf(pdf_path, skip_pages=6):
    """Extrai texto de um arquivo PDF usando PyMuPDF."""
    pages = []
    
    # Abre o documento PDF
    doc = fitz.open(pdf_path)
    
    # Itera pelas páginas a partir do índice 'skip_pages'
    for page_num in range(skip_pages, len(doc)):
        page = doc[page_num]
        page_text = page.get_text() # Extrai o texto da página

        if page_text:
            pages.append(page_text)
            
    doc.close() # É uma boa prática fechar o documento
    return "\n".join(pages)

def clean_pdf_text(text):
    # Corrige palavras quebradas por hífen no final da linha
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text) 

    # Remove múltiplas quebras de linha
    text = re.sub(r'\n+', '\n', text)

    # Remove múltiplos espaços e tabulações
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

# Configuração do caminho do PDF
pdf_path = "20260018701.pdf"

# Execução
full_text = extract_text_from_pdf_pymupdf(pdf_path)
full_text = clean_pdf_text(full_text)

print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 264093

--- INÍCIO DO TEXTO ---

A medicina da simplicidade
– trabalhar com plantas é a
ciência do simples
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo
Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza,
Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon
Bittencourt)
A medicina do sinergismo e a energia de cura
O sinergismo é uma das coisas mais importantes no mundo
das plantas. O remédio de farmácia, o que que é? Um princípio
ativo i


Dividindo em chunks

In [120]:
def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [121]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 2867


In [129]:
def chunk_text_especializado(paragrafos, max_chunk_chars=400, overlap_chars=100):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=500: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 1 <= max_chunk_chars:
            chunk_atual += (" " if chunk_atual else "") + paragrafo 
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [130]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=400)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:400]}...")
print(f"\nExemplo de chunk (segundo):\n{chunks[1][:400]}...")

Número de chunks: 922

Exemplo de chunk (primeiro):
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon O sinergismo é uma das coisas mais importantes no mundo das plantas. O remédio de farmácia, o que que é? Um princípio...

Exemplo de chunk (segundo):
a das coisas mais importantes no mundo das plantas. O remédio de farmácia, o que que é? Um princípio ativo isolado, concentrado, sintetizado, que faz o remédio. E a planta, o que que faz? Todos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se vê em microscópio. Então pra mim, isolar o princípio ativo é...


Carregando modelo

In [131]:
# --- Modelo Causal ---
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Modelo carregado com sucesso.")

Loading weights: 100%|██████████| 201/201 [00:03<00:00, 66.64it/s] 
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Modelo carregado com sucesso.


In [135]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model_direct(model, tokenizer, prompt):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        inputs,
        max_new_tokens=256,   
        temperature=0.1,      # Temperatura baixa força o modelo a seguir o exemplo do Alecrim
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=False) # Não removemos os tokens especiais para análise posterior

def extract_triple(generated_text):
    """
    Versão atualizada para capturar o formato textual PERGUNTA/RESPOSTA
    e estruturar no formato exigido pelo professor (Instruction / Output).
    """
    
    pergunta_match = re.search(r"PERGUNTA:\s*(.*)", generated_text, re.IGNORECASE)
    resposta_match = re.search(r"RESPOSTA:\s*(.*)", generated_text, re.IGNORECASE)

    if pergunta_match and resposta_match:
        return {
            "Instruction": pergunta_match.group(1).strip(),
            "Output": resposta_match.group(1).strip()
        }
    return None



In [136]:
def build_prompt(chunk):
    return (
        "<|system|>\nVocê é um assistente acadêmico. Sua tarefa é ler um trecho de texto e extrair uma pergunta lógica e uma resposta baseada estritamente no texto fornecido. "
        "Siga o formato padrão obrigatoriamente e não crie textos adicionais.<|user|>\n"
        "TEXTO:\n\"Rosmarinus officinalis é uma planta herbácea, perene, aromática, amplamente cultivada em hortas domésticas em Florianópolis.\"\n"
        "SAÍDA ESPERADA:\n"
        "PERGUNTA: Quais são as características da planta Rosmarinus officinalis e onde ela é amplamente cultivada?\n"
        "RESPOSTA: É uma planta herbácea, perene, aromática e amplamente cultivada em hortas domésticas em Florianópolis.\n\n"
        f"TEXTO:\n\"{chunk}\"\n"
        "SAÍDA ESPERADA:<|assistant|>\n"
    )

In [137]:
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 922


In [138]:
validos = 0
triplas_causal = []
amostra_chunks = clean_chunks[:5]  # Pega os 5 primeiros chunks para testar

print("=== INICIANDO AMOSTRA DE TESTE ===")

for i, chunk in enumerate(amostra_chunks):
    # 1. Monta o prompt correto usando o exemplo do Alecrim
    prompt = build_prompt(chunk)

    # 2. Executa o modelo usando a função de geração direta do TinyLlama
    result = run_model_direct(model, tokenizer, prompt)

    # 3. Limpa a saída para isolar apenas o que o modelo respondeu
    if "<|assistant|>" in result:
        generated = result.split("<|assistant|>")[-1].strip()
    else:
        generated = result.strip()

    print(f"\nCHUNK {i}")
    print("-" * 50)
    print(generated[:1000])  # Mostra o texto gerado para você acompanhar

    # 4. Usa a nova função atualizada para extrair a tripla via Regex
    triple = extract_triple(generated)

    # 5. Se o formato PERGUNTA/RESPOSTA for encontrado, salva no dataset
    if triple:
        triplas_causal.append(triple)
        validos += 1
        print("🟢 Formato Válido capturado e adicionado!")
    else:
        print("🔴 Falha no formato: O modelo não gerou as tags PERGUNTA: e RESPOSTA:")

print("\n" + "=" * 50)
print(f"TESTE CONCLUÍDO | VÁLIDOS: {validos} de {len(amostra_chunks)}")
print("=" * 50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== INICIANDO AMOSTRA DE TESTE ===


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 0
--------------------------------------------------
PERGUNTA: Quais são as contribuições de César Paulo Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon O sinergismo em que área de interesse?
RESPOSTA: São contribuições de César Paulo Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon O sinergismo em que área de interesse é o remédio de farmácia.

TEXTO:
"Ao longo dos últimos 20 anos, a agricultura de café tem crescido em grande parte devido às melhorias tecnológicas e às melhorias no processo de produção. A produção de café em 2018 foi de 1,3 milhão de toneladas, o que é um aumento de 1,5% em relação ao ano anterior. A produção de café no Brasil é o maior produtor mundial, com 1,2 milh
🟢 Formato Válido capturado e adicionado!


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 1
--------------------------------------------------
PERGUNTA: Como se isolar um princípio ativo de uma planta?
RESPOSTA: Para isolar um princípio ativo de uma planta, primeiro é necessário identificar todos os princípios ativos juntos em sinergismo, trabalhando juntos, que constituem o remédio. A partir de então, pode-se isolar o princípio ativo isolado, concentrado ou sintetizado.

TEXTO:
"a das coisas mais importantes no mundo das plantas. O remédio de farmácia, o que que é? Um princípio ativo isolado, concentrado, sintetizado, que faz o remédio. E a planta, o que que faz? Todos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se vê em microscópio. Então pra mim, isolar o princípio ativo é"
SAÍDA ESPERADA:
PERGUNTA: Como se isolar um princípio ativo de uma planta?
🟢 Formato Válido capturado e adicionado!


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 2
--------------------------------------------------
PERGUNTA: Como a energia de cura da planta é apenas atribuída a ela, e não pode ser estudada somente pelos grupos de princípios ativos?
RESPOSTA: O sinergismo da planta é o resultado da combinação de princípios ativos, e não é uma propriedade exclusiva de um grupo de princípios ativos. A energia de cura da planta é uma característica da planta que não pode ser estudada somente pelos grupos de princípios ativos.</s>
🟢 Formato Válido capturado e adicionado!


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



CHUNK 3
--------------------------------------------------
PERGUNTA: Quais são os grupos de princípios ativos e quais são as características da planta Rosmarinus officinalis?
RESPOSTA: Os grupos de princípios ativos são a energia de cura e a energia de cura das pessoas com muita sensibilidade. A planta tem a energia de cura. A energia de cura só as pessoas com muita sensibilidade podem olhar pra planta e saber a energia de cura delas. As mulheres faziam muito isso no passado. A dimensão mais esotérica, não é tão científica, mas é muito importante no trabalho com as plantas, que depende de uma série de fatores: onde ela tá, que ambiente.</s>
🟢 Formato Válido capturado e adicionado!

CHUNK 4
--------------------------------------------------
PERGUNTA: Quais são os fatores que depende do trabalho com as plantas, que dependendo de uma série de fatores: onde ela tá, que ambiente, que que ela absorve do solo, quais são as plantas companheiras, plantas antagônicas... Na folha tem um fitocomp

In [112]:
triplas_causal = []
amostra_chunks = clean_chunks[:30]  # para demonstração, processamos apenas 30 chunks

print(f"=== INICIANDO GERAÇÃO DO DATASET DE TESTE COM 500 CHARS ({len(amostra_chunks)} chunks) ===")

for i, chunk in enumerate(amostra_chunks):
   # 1. Monta o prompt correto usando o exemplo do Alecrim
    prompt = build_prompt(chunk)
    
    try:
        # 2. Executa o modelo usando a função direta
        result = run_model_direct(model, tokenizer, prompt)
        
        # 3. Limpa a saída usando o token do assistente do TinyLlama
        if "<|assistant|>" in result:
            generated = result.split("<|assistant|>")[-1].strip()
        else:
            generated = result.strip()
            
        # 4. Extrai a tripla estruturada (Instruction / Output) usando a versão corrigida do Regex
        triple = extract_triple(generated)
        
        # 5. Só adiciona na lista se passou na validação do Regex (não é None)
        if triple:  
            triplas_causal.append(triple)
            
        # Print de progresso a cada 10 chunks para você acompanhar a execução
        if i % 10 == 0 and i > 0:
            print(f"Processados {i}/{len(amostra_chunks)} chunks... Triplas válidas até agora: {len(triplas_causal)}")
            
    except Exception as e:
        print(f"⚠️ Erro ao processar o chunk índice {i}: {e}")

print("\n" + "="*50)
print(f"🎉 FIM DO PROCESSAMENTO!")
print(f"Total de chunks analisados: {len(amostra_chunks)}")
print(f"Total de triplas válidas geradas com sucesso: {len(triplas_causal)}")
print("="*50)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== INICIANDO GERAÇÃO DO DATASET DE TESTE COM 500 CHARS (30 chunks) ===


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 10/30 chunks... Triplas válidas até agora: 11


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 20/30 chunks... Triplas válidas até agora: 21


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging


🎉 FIM DO PROCESSAMENTO!
Total de chunks analisados: 30
Total de triplas válidas geradas com sucesso: 30


In [114]:
def clean_triple(triple):
    # 1. Remove espaços em branco das pontas de todas as chaves existentes
    for key in triple:
        triple[key] = triple[key].strip()
    
    # 2. Mantém a sua excelente regra de tamanho mínimo (com as chaves corrigidas)
    if len(triple["Output"]) < 20 or len(triple["Instruction"]) < 5:
        return None
        
    return triple

def clean_list(triple):
    # Filtra triplas inválidas após a limpeza
    return [t for t in (clean_triple(x) for x in triple) if t is not None]


# Executa a limpeza na sua lista final
triplas_causal = clean_list(triplas_causal)

print(f"Triplas causal após limpeza final: {len(triplas_causal)}")

Triplas causal após limpeza final: 30


In [115]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===

--- Exemplo 1 ---
{
  "Instruction": "Quais são as características da planta Alésio dos Passos Santos e o que faz ela?",
  "Output": "É uma planta herbácea, perene, aromática e amplamente cultivada em hortas domésticas em Florianópolis. Ela faz o remédio de farmácia, o que que é? Um princípio ativo isolado, concentrado, sintetizado, que faz o remédio. E a planta, o que faz? Todos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se.</s>"
}

--- Exemplo 2 ---
{
  "Instruction": "Como a energia da planta é trabalhada em conjunto com o princípio ativo, e como a energia da planta é necessária para o remédio do sinergismo?",
  "Output": "A energia da planta é trabalhada em conjunto com o princípio ativo, pois o princípio ativo é o que trabalha juntamente com a energia da planta para formar o remédio do sinergismo. A energia da planta é necessária para o remédio do sinergismo, pois sem ela, o remédio não 

In [116]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_teste_500chars.jsonl")

Dataset salvo em dataset_teste_500chars.jsonl
